# Process and View Results From Multiple Longstrips

In [1]:
#%% imports
from pathlib import Path
# import json
# import configparser
import pprint
pp = pprint.PrettyPrinter(indent=4);
import time
import math
from collections import namedtuple

# # pip install slicerio
# import slicerio.server

# # packages for 3d
# #   # for OCT 3D stuff and using in jupyter
# #   - trame-jupyter-extension
# #   - trame
# #   - trame-vtk
# #   - trame-vuetify
# #   - ipywidgets

import matplotlib.pyplot as plt
import plotly.express as px

# interactive panels
import panel as pn
pn.extension('plotly');

import numpy as np
import pandas as pd
import addict

import cv2
import skimage
import scipy.signal
import networkx as nx

import pyvista as pv
from PIL import Image
import SimpleITK as sitk

# with vedo
#from vedo import dataurl, Volume, Text2D
import vedo
vedo.settings.default_backend = 'vtk'
#from vedo.applications import Slicer3DPlotter
import naatos_oct_tools.plotters.vedo_plotters as vedo_plotters
import naatos_oct_tools.plotters.sitk_plotters as sitk_plotters

import naatos_oct_tools.thorlabs_oct_file_reading
import naatos_oct_tools.oct_linear_scan_processing

In [2]:
# Magics to autoreload submodules when they are modified
%load_ext autoreload
%autoreload 2

In [8]:
#%% Test Record Excel
dftests = pd.read_excel(
    r'C:\Users\SimonGhionea\Global Health Labs, Inc\NAATOS Product Feasibility - TB V1 - General - Internal - Wax Valve\OCT\Test\OCT_wax_valve_test_record.xlsx',
    skiprows=1
)
dftests = dftests.iloc[:,1:]
dftests.rename(columns={'Unnamed: 19':'Notes'},inplace=True)
dftests

,Test ID,Test date,Test name,Batch,Strip,Wax (mg/mm),Fixture,"X, FOV","Y, FOV","Z, FOV",...,Right edge (mm),Left edge (mm),Strip length (mm),Speed/Sensitivity,Averaging (A-scan),Refractive Index,File size (MB),Total size (GB),ProcessingNotes,Notes
0,NaN,2025-03-26,GHL_pyapp_20250326T1258,oven aging test day 1,strip 7,NaN,"old, with tape",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,2025-03-26,GHL_pyapp_20250326T1403,oven aging test day 1,strip 10,NaN,"old, with tape",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,2025-03-26,GHL_pyapp_20250326T1416,oven aging test day 1,strip 14,NaN,"old, with tape",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,2025-03-27,GHL_pyapp_20250327T1428,oven aging test day 2,strip 7,NaN,"old, with 3d print clamps",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,2025-03-27,GHL_pyapp_20250327T1438,oven aging test day 2,strip 10,NaN,"old, with 3d print clamps",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118,113.0,2025-06-05,GHL_pyapp_20250605T1621,Jun5 0.375-0.400mg/mm bag blu crossouts,ES0403#4 0.392,0.392,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"done,1",NaN
119,114.0,2025-06-05,GHL_pyapp_20250605T1625,Jun5 0.375-0.400mg/mm bag blu crossouts,ES0403#1BluMark 0.382,0.382,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"done,2",NaN
120,115.0,2025-06-05,GHL_pyapp_20250605T1628,Jun5 0.375-0.400mg/mm bag blu crossouts,ES0403#7 0.383,0.383,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"done,3",NaN
121,116.0,2025-06-05,GHL_pyapp_20250605T1633,Jun5 0.375-0.400mg/mm bag blu crossouts,ES0402#14 0.384,0.384,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"done,1",NaN


In [10]:
#%% Filter The Tests To View
# dfmasks = [
#     (dftests['Test date']<'2025-03-28') & (dftests['Test date']>='2025-03-26'),
#     dftests['Test name'] != 'GHL_pyapp_20250326T1258'
# ]
# dfmasks = [
#     (dftests['Test date']<='2025-04-15') & (dftests['Test date']>='2025-04-14')
# ]
# dfmasks = [
#     (dftests['Test date']=='2025-05-05')
# ]
dfmasks = [
    dftests['Test date']=='2025-06-05',
    #dftests['Test name']!='GHL_pyapp_20250512T',
    #dftests['ProcessingNotes'].str.startswith('done,2'),
    #~dftests['Batch'].str.contains('Valve')
]

dffilt = dftests[np.all(dfmasks,axis=0)]
dffilt

,Test ID,Test date,Test name,Batch,Strip,Wax (mg/mm),Fixture,"X, FOV","Y, FOV","Z, FOV",...,Right edge (mm),Left edge (mm),Strip length (mm),Speed/Sensitivity,Averaging (A-scan),Refractive Index,File size (MB),Total size (GB),ProcessingNotes,Notes
74,69.0,2025-06-05,GHL_pyapp_20250605T1211,Jun5 0.325-0.349mg/mm bag ylw,BluePenES0323#16 0.335,0.335,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"done,2",NaN
75,70.0,2025-06-05,GHL_pyapp_20250605T1214,Jun5 0.325-0.349mg/mm bag ylw,ES0403#8 0.326,0.326,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"done,1",NaN
76,71.0,2025-06-05,GHL_pyapp_20250605T1218,Jun5 0.325-0.349mg/mm bag ylw,ES0402#11 0.342,0.342,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"done,1",NaN
77,72.0,2025-06-05,GHL_pyapp_20250605T1225,Jun5 0.325-0.349mg/mm bag ylw,ES0403#6 0.331,0.331,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"done,1",NaN
78,73.0,2025-06-05,GHL_pyapp_20250605T1228,Jun5 0.325-0.349mg/mm bag ylw,ES0403#19 0.329,0.329,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"done,1",NaN
79,74.0,2025-06-05,GHL_pyapp_20250605T1233,Jun5 0.325-0.349mg/mm bag ylw,ES0403#21 0.340,0.340,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"done,1",NaN
80,75.0,2025-06-05,GHL_pyapp_20250605T1301,Jun5 0.325-0.349mg/mm bag ylw,ES0403#30 0.345,0.345,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"done,2",NaN
81,76.0,2025-06-05,GHL_pyapp_20250605T1326,Jun5 0.325-0.349mg/mm bag ylw,ES0407#2 0.326,0.326,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"done,1",NaN
82,77.0,2025-06-05,GHL_pyapp_20250605T1330,Jun5 0.325-0.349mg/mm bag ylw,ES0331#31 0.334,0.334,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"done,1",NaN
83,78.0,2025-06-05,GHL_pyapp_20250605T1333,Jun5 0.325-0.349mg/mm bag ylw,ES0331#35 0.327,0.327,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"done,3",NaN


In [11]:
#%% Load OCT study information
octstudies = [];
folder_octexport_root = Path(r'\\file.corp.ghlabs.org\Shared\Projects\NAATOS\V1\NAATOS_OCT_WORK\OCTExport')
#folder_octexport_root = Path(r'D:\SGProjects\NAATOS\OCTlocal')

folder_temp = Path(r'D:\TEMP')

In [12]:
# Load list of data
for idx,record in dffilt.iterrows():
    octstudy = naatos_oct_tools.thorlabs_oct_file_reading.OCT_Study_Folder(record['Test name'],folder_octexport_root);
    octstudies.append(octstudy);

STUDY: GHL_pyapp_20250605T1211
{   'study_has_an_ini_file': True,
    'study_has_json_info_file': True,
    'study_has_yat_log': False,
    'study_num_jpg_files': 0,
    'study_num_oct_files': 19,
    'study_num_vtk_files': 0}
STUDY: GHL_pyapp_20250605T1214
{   'study_has_an_ini_file': True,
    'study_has_json_info_file': True,
    'study_has_yat_log': False,
    'study_num_jpg_files': 0,
    'study_num_oct_files': 19,
    'study_num_vtk_files': 0}
STUDY: GHL_pyapp_20250605T1218
{   'study_has_an_ini_file': True,
    'study_has_json_info_file': True,
    'study_has_yat_log': False,
    'study_num_jpg_files': 0,
    'study_num_oct_files': 19,
    'study_num_vtk_files': 0}
STUDY: GHL_pyapp_20250605T1225
{   'study_has_an_ini_file': True,
    'study_has_json_info_file': True,
    'study_has_yat_log': False,
    'study_num_jpg_files': 0,
    'study_num_oct_files': 19,
    'study_num_vtk_files': 0}
STUDY: GHL_pyapp_20250605T1228
{   'study_has_an_ini_file': True,
    'study_has_json_info_f

# Parse Through The OCTSTUDIES

In [13]:
#%% Load OCTSTUDY object, and start some processing on it
octstudies_to_run = [];
for idx,octstudy in enumerate(octstudies):
    octstudy : naatos_oct_tools.thorlabs_oct_file_reading.OCT_Study_Folder;
    print('~~~~~~~');
    print(octstudy.name);
    pp.pprint(octstudy.resultsCheck())
    doWeRunTheNotebook = any([v is False for k,v in octstudy.resultsCheck().items()])
    print('Run?',doWeRunTheNotebook)
    if(not doWeRunTheNotebook):
        octstudies_to_run.append(octstudy);

    print('~~~~~~~');
    print('We will analyze data for {:} octstudies.'.format(len(octstudies_to_run)))
    print([x.name for x in octstudies_to_run]);

~~~~~~~
GHL_pyapp_20250605T1211
{   'along_strip_data_extracted': [   '/dfstepA',
                                      '/dfstepB',
                                      '/dfstepC',
                                      '/dfstepD',
                                      '/dfstepE'],
    'exist_data_extracted': True,
    'figoutmp4_mazetest': True,
    'figoutmp4_stepA': True,
    'figoutmp4_stepB': True}
Run? False
~~~~~~~
We will analyze data for 1 octstudies.
['GHL_pyapp_20250605T1211']
~~~~~~~
GHL_pyapp_20250605T1214
{   'along_strip_data_extracted': [   '/dfstepA',
                                      '/dfstepB',
                                      '/dfstepC',
                                      '/dfstepD',
                                      '/dfstepE'],
    'exist_data_extracted': True,
    'figoutmp4_mazetest': True,
    'figoutmp4_stepA': True,
    'figoutmp4_stepB': True}
Run? False
~~~~~~~
We will analyze data for 2 octstudies.
['GHL_pyapp_20250605T1211', 'GHL_pyapp_202

# 1. Load all Data

In [14]:
#octstudies_to_run = [octstudies_to_run[0]]
dfs = [];
for idx,octstudy in enumerate(octstudies_to_run):
    print(octstudy.name)

    #octstudy.load_previously_saved_merged_volume();

    data_extracted = octstudy.load_data_extracted();
    dfstep = octstudy.load_data_extracted_along_strip();
    
    dfstep['octstudy'] = octstudy;

    dfs.append(dfstep)
    #data_extracted = 


    # Calculate Path From Top To Bottom
dfsteps = pd.concat(dfs,keys=[x.name for x in octstudies_to_run]);

GHL_pyapp_20250605T1211
Loading /dfstepA
Loading /dfstepB
Loading /dfstepC
Loading /dfstepD
Loading /dfstepE
GHL_pyapp_20250605T1214
Loading /dfstepA
Loading /dfstepB
Loading /dfstepC
Loading /dfstepD
Loading /dfstepE
GHL_pyapp_20250605T1218
Loading /dfstepA
Loading /dfstepB
Loading /dfstepC
Loading /dfstepD
Loading /dfstepE
GHL_pyapp_20250605T1225
Loading /dfstepA
Loading /dfstepB
Loading /dfstepC
Loading /dfstepD
Loading /dfstepE
GHL_pyapp_20250605T1228
Loading /dfstepA
Loading /dfstepB
Loading /dfstepC
Loading /dfstepD
Loading /dfstepE
GHL_pyapp_20250605T1233
Loading /dfstepA
Loading /dfstepB
Loading /dfstepC
Loading /dfstepD
Loading /dfstepE
GHL_pyapp_20250605T1301
Loading /dfstepA
Loading /dfstepB
Loading /dfstepC
Loading /dfstepD
Loading /dfstepE
GHL_pyapp_20250605T1326
Loading /dfstepA
Loading /dfstepB
Loading /dfstepC
Loading /dfstepD
Loading /dfstepE
GHL_pyapp_20250605T1330
Loading /dfstepA
Loading /dfstepB
Loading /dfstepC
Loading /dfstepD
Loading /dfstepE
GHL_pyapp_20250605T

In [9]:
dfsteps

pixel_depth_strip_top_sum_threshold  \
                        slice                                        
GHL_pyapp_20250508T1112 5                                57.760000   
                        10                               56.800000   
                        15                               59.520000   
                        20                               56.600000   
                        25                               58.080000   
...                                                            ...   
GHL_pyapp_20250512T1303 8240                             51.040000   
                        8245                             51.560000   
                        8250                             55.440000   
                        8255                             54.120000   
                        8260                             53.866667   

                               pixel_depth_strip_top  pixel_depth_strip_bot  \
                        slice                                                 
GHL_pyapp_20250508T1112 5                        181                    247   
                        10                       182                    246   
                        15                       183                    244   
                        20                       184                    244   
                        25                       184                    243   
...                                              ...                    ...   
GHL_pyapp_20250512T1303 8240                     270                    331   
                        8245                     270                    330   
                        8250                     270                    328   
                        8255                     268                    326   
                        8260                     268                    324   

                               pixel_depth_wax_center  \
                        slice                           
GHL_pyapp_20250508T1112 5                         222   
                        10                        223   
                        15                        226   
                        20                        231   
                        25                        223   
...                                               ...   
GHL_pyapp_20250512T1303 8240                      303   
                        8245                      303   
                        8250                      324   
                        8255                      305   
                        8260                      297   

                                                px_wax_transverse_edges  \
                        slice                                             
GHL_pyapp_20250508T1112 5       (56.33712121212121, 105.00769230769231)   
                        10      (56.755319148936174, 105.3936170212766)   
                        15       (56.82631578947368, 103.9396551724138)   
                        20      (55.68604651162791, 102.38636363636364)   
                        25      (53.572463768115945, 102.2843137254902)   
...                                                                 ...   
GHL_pyapp_20250512T1303 8240               (45.228873239436616, 121.35)   
                        8245   (44.877272727272725, 118.78289473684211)   
                        8250              (44.430851063829785, 116.775)   
                        8255    (44.61702127659574, 112.73170731707317)   
                        8260   (44.353932584269664, 112.84482758620689)   

                              wax_top_seed_candidate_px  \
                        slice                             
GHL_pyapp_20250508T1112 5                     (190, 80)   
                        10                    (195, 80)   
                        15                    (202, 79)   
                        20                    (203, 78)   
                    

In [15]:
import gc
gc.collect()

0

## 2. Save

In [16]:
cols_ignore = ['testseg_otsu_thresholds','testseg_otsu_regions','seg_thresh_sauvola_cutoff','pksA','pksB','pksB2']

df = dfsteps[[x for x in dfsteps.columns.tolist() if x not in cols_ignore]]
df.to_hdf(folder_temp/'oct_results_summary_2025_06_10.hdf',key='table');

C:\Users\SimonGhionea\AppData\Local\Temp\ipykernel_49460\4046468293.py:4: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed,key->block2_values] [items->Index(['px_wax_transverse_edges', 'wax_top_seed_candidate_px', 'octstudy'], dtype='object')]

  df.to_hdf(folder_temp/'oct_results_summary_2025_06_10.hdf',key='table');


# handle the source image

In [ ]:
def getSitkImage(octstudy_with_loaded_vdvol):
    # Test getting SITK image
    # Get ITK image without requiring new memory
    import itk
    import vtk

    # function to go from itk image to simpleitk image
    def itkToSimpleITK(itk_image):
        new_sitk_image = sitk.GetImageFromArray(itk.GetArrayViewFromImage(itk_image),isVector=itk_image.GetNumberOfComponentsPerPixel()>1);
        new_sitk_image.SetOrigin(tuple(itk_image.GetOrigin()))
        new_sitk_image.SetSpacing(tuple(itk_image.GetSpacing()))
        new_sitk_image.SetDirection(itk.GetArrayFromMatrix(itk_image.GetDirection()).flatten()) 
        return new_sitk_image;

    print('Get volume for',octstudy_with_loaded_vdvol.name)

    # create simpleitk 3dimage, directly from the previously-loaded vtk 3dimage
    simgmerged = itkToSimpleITK( itk.image_from_vtk_image(octstudy_with_loaded_vdvol.vdvol.dataset) );



    # itk_image
    #print(itk_image)
    slicer_1 = slice( data_extracted['auto_bounds']['topdown_edge_left_px3']  ,  data_extracted['auto_bounds']['topdown_edge_right_px3']  )
    #print(' slicer_dim_0',slicer_0)
    print(' slicer_dim_1',slicer_1)
    #simgmerged.GetSize()
    return simgmerged,{'slicer_1':slicer_1};


simgmerged,voldata = getSitkImage(octstudy);
slicer_1 = voldata['slicer_1'];

In [ ]:
print( simgmerged[:,slicer_1,:] )

In [ ]:
slab_thickness = 10; # px, this was used in the bulk analysis
slice_position_along_strip_px = 4000;
slab_slicer = slice(slice_position_along_strip_px-slab_thickness//2,slice_position_along_strip_px+slab_thickness//2);
simg_slab_img = (simgmerged[:,slicer_1,:])[:,slab_slicer,:]

# 3. Plot Of Values Along Strip

In [ ]:
slice_centers = dfsteps.index;

In [ ]:
simgmerged_summed = sitk.SumProjection(simgmerged[:,slicer_1,:],projectionDimension=0);


In [ ]:
print(simgmerged_summed)

In [ ]:
ndaimg = sitk.GetArrayFromImage(sitk.RescaleIntensity(simgmerged_summed,0,255)); # numpy array


In [ ]:
ndaimg

In [ ]:
#ndaimg = sitk.GetArrayViewFromImage(simgmerged_summed); # numpy array
from PIL import Image
import base64
from io import BytesIO
pil_img = Image.fromarray(ndaimg[:,:,0]) # PIL image object

In [ ]:

ndaimg = sitk.GetArrayViewFromImage(simgmerged_summed); # numpy array
from PIL import Image
import base64
from io import BytesIO
pil_img = Image.fromarray(ndaimg[:,:,0]) # PIL image object

In [ ]:
pil_img.show()

In [ ]:
studyname = dfsteps.index.get_level_values(0).unique().tolist()[1];
octstudy = dfsteps.loc[studyname].iloc[0]['octstudy']

simgmerged,voldata = getSitkImage(octstudy);
slicer_1 = voldata['slicer_1'];
df = dfsteps.loc[studyname,:]

In [ ]:
# Make Detailed Metrics Plots Of 1-Device
import plotly.graph_objects as go
from plotly.subplots import make_subplots

nrows = 5
fig = make_subplots(rows=nrows,shared_xaxes=True,vertical_spacing=0.02, )

slice_centers = df.index;

myrow = 1;
if True:
    # make image
    legendgroup = 'image';

    # use previously created simpleitk merged image filter
    #ndaimg = sitk.GetArrayViewFromImage(simgmerged_summed); # numpy array
    ndaimg = sitk.GetArrayFromImage(sitk.RescaleIntensity(simgmerged_summed,0,255)); # numpy array
    print(ndaimg.shape)
    from PIL import Image
    import base64
    from io import BytesIO
    pil_img = Image.fromarray(ndaimg[:,:,0]) # PIL image object
    pil_img = pil_img.convert("L"); # grayscale image convert, from float
    prefix = "data:image/png;base64,"
    with BytesIO() as stream:
        pil_img.save(stream, format="png")
        base64_string = prefix + base64.b64encode(stream.getvalue()).decode("utf-8")

    fig.add_trace(
        go.Image(
            source=base64_string,
            #cmap='jet',
        ),
        row=myrow,col=1,
    )

legendgroup = 'distances';
myrow+=1;
# Wax Thickness
wax_top = df.loc[slice_centers]['wax_top_seed_candidate_px'].apply(lambda x: x[0]);
#wax_top = dfstepA.loc[final_wax_centers]['pixel_depth_strip_top']
wax_bot = df.loc[slice_centers]['pixel_depth_strip_top']+df.loc[slice_centers]['final_wax_bot_subset_px']
wax_thickness = wax_bot-wax_top
fig.add_trace(
    go.Scatter(
        x = slice_centers,
        y = wax_thickness,
        #type = 'heatmap',
        #colorscale = 'jet'
        name='wax_thickness1',
        legendgroup=legendgroup
    ),
    row=myrow,col=1,
)
#fig.update_yaxes(title='Wax Valve Width (pixels)',col=1);


# Wax Thickness
wax_top = df.loc[slice_centers]['wax_top_seed_candidate_px'].apply(lambda x: x[0]);
#wax_top = dfstepA.loc[final_wax_centers]['pixel_depth_strip_top']
wax_bot = df.loc[slice_centers]['pixel_depth_strip_top']+df.loc[slice_centers]['final_wax_bot_subset_px']
wax_thickness = wax_bot-df.loc[slice_centers]['pixel_depth_strip_top']
fig.add_trace(
    go.Scatter(
        x = slice_centers,
        y = wax_thickness,
        #type = 'heatmap',
        #colorscale = 'jet'
        name='wax_thickness2',
        legendgroup=legendgroup
    ),
    row=myrow,col=1,
)

fig.add_trace(
    go.Scatter(
        x = slice_centers,
        y = df['px_wax_transverse_edges'].apply(lambda x: np.diff(x)[0]),
        #type = 'heatmap',
        #colorscale = 'jet'
        name='wax_width_px',
        legendgroup=legendgroup
    ),
    row=myrow,col=1,
)

fig.add_trace(
    go.Scatter(
        x = slice_centers,
        y = wax_top,
        #type = 'heatmap',
        #colorscale = 'jet'
        name='wax_top',
        legendgroup=legendgroup
    ),
    row=myrow,col=1,
)




legendgroup = 'areas';
myrow+=1;
fig.add_trace(
    go.Scatter(
        x = slice_centers,
        y = df['area_filled']-df['area'],
        #type = 'heatmap',
        #colorscale = 'jet'
        name='seg_area_holes',
        legendgroup=legendgroup
    ),
    row=myrow,col=1,
)

fig.add_trace(
    go.Scatter(
        x = slice_centers,
        y = df['area'],
        #type = 'heatmap',
        #colorscale = 'jet'
        name='seg_area',
        legendgroup=legendgroup
    ),
    row=myrow,col=1,
)

fig.add_trace(
    go.Scatter(
        x = slice_centers,
        y = df['area_filled'],
        #type = 'heatmap',
        #colorscale = 'jet'
        name='seg_area_filled',
        legendgroup=legendgroup
    ),
    row=myrow,col=1,
)

fig.add_trace(
    go.Scatter(
        x = slice_centers,
        y = df['area_convex'],
        #type = 'heatmap',
        #colorscale = 'jet'
        name='seg_area_convex',
        legendgroup=legendgroup
    ),
    row=myrow,col=1,
)


legendgroup = 'ratios';
myrow+=1;
fig.add_trace(
    go.Scatter(
        x = slice_centers,
        y = df['area']/df['area_filled'],
        #type = 'heatmap',
        #colorscale = 'jet'
        name='seg_area_to_areafilled_ratio',
        legendgroup=legendgroup
    ),
    row=myrow,col=1,
)
fig.add_trace(
    go.Scatter(
        x = slice_centers,
        y = df['area']/df['area_convex'],
        #type = 'heatmap',
        #colorscale = 'jet'
        name='seg_area_to_areaconvex_ratio',
        legendgroup=legendgroup
    ),
    row=myrow,col=1,
)
fig.add_trace(
    go.Scatter(
        x = slice_centers,
        y = df['area_filled']/df['area_convex'],
        #type = 'heatmap',
        #colorscale = 'jet'
        name='seg_areafilled_to_areaconvex_ratio',
        legendgroup=legendgroup
    ),
    row=myrow,col=1,
)

legendgroup = 'counts';
myrow+=1;
fig.add_trace(
    go.Scatter(
        x = slice_centers,
        y = df['num_paths'],
        name='crossovers_px_paths',
        legendgroup=legendgroup,
    ),
    row=myrow,col=1,
)
fig.update_yaxes(row=myrow,type='log')

# setup legends per row
for i, yaxis in enumerate(fig.select_yaxes(col=1), 1):
    legend_name = f"legend{i}"
    fig.update_layout({legend_name: dict(y=yaxis.domain[1], yanchor="top")}, showlegend=True)
    fig.update_traces(row=i, legend=legend_name)

fig.update_traces(xaxis='x{:}'.format(nrows))
fig.update_yaxes(title='pixels',row=1);
fig.update_yaxes(title='pixels^2',row=2);
fig.update_yaxes(title='ratios',row=3);
fig.update_yaxes(title='counts',row=4);
fig.update_xaxes(title='distance along strip (px)',row=nrows,col=1);
fig.update_layout(hovermode='x unified',hoversubplots="axis",spikedistance=-1);
#fig = px.imshow(sum_wax_strip_along_length[:,0:]);

fig.show(renderer='browser')
#fig

In [ ]:
fig.update_yaxes(row=1,scaleanchor=None)

In [ ]:
fig.layout

In [ ]:
fig.layout.yaxis5

# Multi-strip Data Viewers

## 5-14-2025 Now

In [ ]:
dfsteps.info()

In [ ]:
from dataclasses import dataclass

# @dataclass
# class row_traces:
#     ylabel : str
#     column_names

In [ ]:
#cols_ignore = ['testseg_otsu_thresholds','testseg_otsu_regions','seg_thresh_sauvola_cutoff','pksA','pksB','pksB2','octstudy']
cols_ignore = ['testseg_otsu_thresholds','testseg_otsu_regions','seg_thresh_sauvola_cutoff','pksA','pksB','pksB2']
dfsteps.iloc[1000][[x for x in dfsteps.columns.tolist() if x not in cols_ignore]]

In [ ]:
cols_ignore = ['testseg_otsu_thresholds','testseg_otsu_regions','seg_thresh_sauvola_cutoff','pksA','pksB','pksB2']

df = dfsteps[[x for x in dfsteps.columns.tolist() if x not in cols_ignore]]
df.to_hdf(folder_temp/'oct_results_summary_2025_06_10.hdf',key='table');

In [ ]:
# strip down columns we will not use below
df = dfsteps[[x for x in dfsteps.columns.tolist() if x not in cols_ignore]];
# join info from spreadsheet
df = pd.merge(df.reset_index(),dffilt,left_on='level_0',right_on='Test name');
df

In [ ]:
dfmelted = pd.melt(frame=df,value_vars=['pixel_depth_strip_top'],id_vars=['Test name','slice'])
dfmelted

In [ ]:

slice_centers = df.index;

# Wax Thickness
wax_top = df.loc[slice_centers]['wax_top_seed_candidate_px'].apply(lambda x: x[0]);
#wax_top = dfstepA.loc[final_wax_centers]['pixel_depth_strip_top']
wax_bot = df.loc[slice_centers]['pixel_depth_strip_top']+df.loc[slice_centers]['final_wax_bot_subset_px']
wax_thickness = wax_bot-wax_top

df['wax_thickness1'] = wax_bot-wax_top;
df['wax_thickness2'] = wax_bot-df.loc[slice_centers]['pixel_depth_strip_top'];
df['wax_width_px'] = df['px_wax_transverse_edges'].apply(lambda x: np.diff(x)[0]);
df['wax_top'] = df['wax_top_seed_candidate_px'].apply(lambda x: x[0]);


df['seg_area'] = df['area'];
df['seg_area_filled'] = df['area_filled'];
df['seg_area_holes'] = df['area_filled']-df['area'];

df['seg_area_to_areafilled'] = df['area']/df['seg_area_filled'];
df['seg_area_to_areaconvex'] = df['area']/df['area_convex'];
df['seg_areafilled_to_areaconvex'] = df['seg_area_filled']/df['area_convex'];


In [ ]:
# # Make Detailed Metrics Plots Of 1-Device
# import plotly.graph_objects as go
# from plotly.subplots import make_subplots

# nrows = 4
# fig = make_subplots(rows=nrows,shared_xaxes=True,vertical_spacing=0.02, )
# #tracelabels

# myrow = 1;
# #px.scatter(df,x='slice',y='wax_thickness1',color='Test name')
# fig1 = px.scatter(df,x='slice',y='wax_width_px',color='Test name')
# fig1.show(renderer='browser')

In [ ]:
trace_labels = [];
trace_info = [];

#trace_colors = [];
import itertools
#cyc_color = itertools.cycle(px.colors.qualitative.Alphabet)
cyc_color = itertools.cycle(px.colors.qualitative.Light24)
df_grouped_by_test = df.groupby('Test name');
for studyname,dfv in df_grouped_by_test:
    print(studyname)
    df1 = dfv.iloc[0];
    
    octstudy = df1['octstudy']

    trace_labels.append( octstudy.name )
    #trace_colors.append( next(cyc_color) );
    trace_info.append(
        dict(
            octstudyname = octstudy.name,
            
            tracelabel = octstudy.name,
            tracecolor = next(cyc_color),
        )
    )

In [ ]:
# Make Detailed Metrics Plots Of 1-Device
import plotly.graph_objects as go
from plotly.subplots import make_subplots

#metrics_to_show = ['wax_width_px','seg_area','seg_area_holes','wax_thickness2'];
metrics_to_show = ['wax_width_px','seg_area_to_areafilled','num_paths'];
nrows = len(metrics_to_show)
fig = make_subplots(rows=nrows,shared_xaxes=True,vertical_spacing=0.02, )

myrow = 0;

#legendgroup = 'distances';
for metric_col_name in metrics_to_show:
    myrow+=1;
    
    if True:
        for this_trace_info in trace_info:
            dfv = df_grouped_by_test.get_group(this_trace_info['octstudyname'])

            datax = dfv['slice'];
            datay = dfv[metric_col_name]

            fig.add_trace(
                go.Scattergl(
                    x = datax,
                    y = datay,
                    mode='markers',
                    #type = 'heatmap',
                    #colorscale = 'jet'
                    name=this_trace_info['tracelabel'],
                    marker_color=this_trace_info['tracecolor'],
                    legendgroup=this_trace_info['tracelabel'],
                    showlegend=myrow==1,
                    hoverinfo='name',
                    visible='legendonly',
                ),
                row=myrow,col=1,
            )
    
    fig.update_yaxes(title=metric_col_name,row=myrow);

    if False:
        figtmp = px.scatter(df,x='slice',y=metric_col_name,color='Test name')
        fig.add_traces(figtmp.data,rows=myrow,cols=1);

    #break;




# # setup legends per row
# for i, yaxis in enumerate(fig.select_yaxes(col=1), 1):
#     legend_name = f"legend{i}"
#     fig.update_layout({legend_name: dict(y=yaxis.domain[1], yanchor="top")}, showlegend=True)
#     fig.update_traces(row=i, legend=legend_name)
fig.update_traces(marker_size=4)
fig.update_layout(legend_tracegroupgap=0)

fig.update_traces(xaxis='x{:}'.format(nrows))

# fig.update_yaxes(title='pixels',row=1);
# fig.update_yaxes(title='pixels^2',row=2);
# fig.update_yaxes(title='ratios',row=3);
# fig.update_yaxes(title='counts',row=4);

fig.update_xaxes(title='distance along strip (px)',row=nrows,col=1);

#fig.update_layout(hovermode='x',hoversubplots="axis",spikedistance=-1);
#fig.update_layout(hovermode='x unified',spikedistance=-1,hoverdistance=5);
fig.update_xaxes(showspikes=True, spikesnap="cursor", spikemode="across")

#fig = px.imshow(sum_wax_strip_along_length[:,0:]);

fig.show(renderer='browser')
#fig

## Single-Device Detail

In [ ]:
#df = df_grouped_by_test.get_group('GHL_pyapp_20250512T1245')
df = df_grouped_by_test.get_group('GHL_pyapp_20250508T1158')
df

In [ ]:
metric_traces = [
    ('distances','pixels',['wax_top','wax_thickness1','wax_thickness2','wax_width_px']),
    ('areas','pixels^2',['area','area_filled','area_convex','seg_area_holes']),
    ('ratios','ratio',['seg_area_to_areafilled','seg_area_to_areaconvex','seg_areafilled_to_areaconvex']),
    ('counts','counts',['num_paths']),
]

In [ ]:
# Make Detailed Metrics Plots Of 1-Device
import plotly.graph_objects as go
from plotly.subplots import make_subplots

nrows = len(metric_traces);
fig = make_subplots(rows=nrows,shared_xaxes=True,vertical_spacing=0.02)

myrow = 0;
for legendgroup,ylabel,metric_columns in metric_traces:
    myrow += 1;

    datax = df['slice'];
    for metric_col in metric_columns:
        datay = df[metric_col];

        if(ylabel!='counts'):
            fig.add_trace(
                go.Scattergl(
                    x = datax,
                    y = datay,
                    mode='markers',
                    #type = 'heatmap',
                    #colorscale = 'jet'
                    name=metric_col,
                    #marker_color=this_trace_info['tracecolor'],
                    legendgroup=legendgroup,
                    legendgrouptitle={'text': legendgroup},
                    #showlegend=myrow==1,
                    #hoverinfo='name',
                    #visible='legendonly',
                ),
                row=myrow,col=1,
            )
        else:
            fig.add_trace(
                go.Bar(
                    x = datax,
                    y = datay,
                    #mode='markers',
                    #type = 'heatmap',
                    #colorscale = 'jet'
                    name=metric_col,
                    #marker_color=this_trace_info['tracecolor'],
                    legendgroup=legendgroup,
                    legendgrouptitle={'text': legendgroup},
                    #showlegend=myrow==1,
                    #hoverinfo='name',
                    #visible='legendonly',
                ),
                row=myrow,col=1,
            )

    
    fig.update_yaxes(title=ylabel,row=myrow);
    if(ylabel=='counts'):
        fig.update_yaxes(type='log',row=myrow);

# setup legends per row
for i, yaxis in enumerate(fig.select_yaxes(col=1), 1):
    legend_name = f"legend{i}"
    fig.update_layout({legend_name: dict(y=yaxis.domain[1], yanchor="top")}, showlegend=True)
    fig.update_traces(row=i, legend=legend_name)

fig.update_xaxes(title='distance along strip (px)',row=nrows,col=1);

fig.update_traces(xaxis='x{:}'.format(nrows))
fig.update_layout(hovermode='x unified',spikedistance=-1,hoverdistance=5);
#fig.update_yaxes(title='pixels',row=1);
fig.show(renderer='browser')

In [ ]:
fig.add_trace(
    go.Scatter(
        x = slice_centers,
        y = df['area']/df['area_filled'],
        #type = 'heatmap',
        #colorscale = 'jet'
        name='seg_area_to_areafilled_ratio',
        legendgroup=legendgroup
    ),
    row=myrow,col=1,
)
fig.add_trace(
    go.Scatter(
        x = slice_centers,
        y = df['area']/df['area_convex'],
        #type = 'heatmap',
        #colorscale = 'jet'
        name='seg_area_to_areaconvex_ratio',
        legendgroup=legendgroup
    ),
    row=myrow,col=1,
)
fig.add_trace(
    go.Scatter(
        x = slice_centers,
        y = df['area_filled']/df['area_convex'],
        #type = 'heatmap',
        #colorscale = 'jet'
        name='seg_areafilled_to_areaconvex_ratio',
        legendgroup=legendgroup
    ),
    row=myrow,col=1,
)

In [ ]:
#ndaimg = sitk.GetArrayViewFromImage(simgmerged_summed); # numpy array
from PIL import Image
import base64
from io import BytesIO
pil_img = Image.fromarray(ndaimg[:,:,0]) # PIL image object

In [ ]:

ndaimg = sitk.GetArrayViewFromImage(simgmerged_summed); # numpy array
from PIL import Image
import base64
from io import BytesIO
pil_img = Image.fromarray(ndaimg[:,:,0]) # PIL image object

In [ ]:
pil_img.show()

In [ ]:
studyname = dfsteps.index.get_level_values(0).unique().tolist()[1];
octstudy = dfsteps.loc[studyname].iloc[0]['octstudy']

simgmerged,voldata = getSitkImage(octstudy);
slicer_1 = voldata['slicer_1'];
df = dfsteps.loc[studyname,:]

In [ ]:
# Make Detailed Metrics Plots Of 1-Device
import plotly.graph_objects as go
from plotly.subplots import make_subplots

nrows = 5
fig = make_subplots(rows=nrows,shared_xaxes=True,vertical_spacing=0.02, )

slice_centers = df.index;

myrow = 1;
if False:
    # make image
    legendgroup = 'image';

    # use previously created simpleitk merged image filter
    #ndaimg = sitk.GetArrayViewFromImage(simgmerged_summed); # numpy array
    ndaimg = sitk.GetArrayFromImage(sitk.RescaleIntensity(simgmerged_summed,0,255)); # numpy array
    print(ndaimg.shape)
    from PIL import Image
    import base64
    from io import BytesIO
    pil_img = Image.fromarray(ndaimg[:,:,0]) # PIL image object
    pil_img = pil_img.convert("L"); # grayscale image convert, from float
    prefix = "data:image/png;base64,"
    with BytesIO() as stream:
        pil_img.save(stream, format="png")
        base64_string = prefix + base64.b64encode(stream.getvalue()).decode("utf-8")

    fig.add_trace(
        go.Image(
            source=base64_string,
            #cmap='jet',
        ),
        row=myrow,col=1,
    )

legendgroup = 'distances';
myrow+=1;
# Wax Thickness
wax_top = df.loc[slice_centers]['wax_top_seed_candidate_px'].apply(lambda x: x[0]);
#wax_top = dfstepA.loc[final_wax_centers]['pixel_depth_strip_top']
wax_bot = df.loc[slice_centers]['pixel_depth_strip_top']+df.loc[slice_centers]['final_wax_bot_subset_px']
wax_thickness = wax_bot-wax_top
fig.add_trace(
    go.Scatter(
        x = slice_centers,
        y = wax_thickness,
        #type = 'heatmap',
        #colorscale = 'jet'
        name='wax_thickness1',
        legendgroup=legendgroup
    ),
    row=myrow,col=1,
)
#fig.update_yaxes(title='Wax Valve Width (pixels)',col=1);


# Wax Thickness
wax_top = df.loc[slice_centers]['wax_top_seed_candidate_px'].apply(lambda x: x[0]);
#wax_top = dfstepA.loc[final_wax_centers]['pixel_depth_strip_top']
wax_bot = df.loc[slice_centers]['pixel_depth_strip_top']+df.loc[slice_centers]['final_wax_bot_subset_px']
wax_thickness = wax_bot-df.loc[slice_centers]['pixel_depth_strip_top']
fig.add_trace(
    go.Scatter(
        x = slice_centers,
        y = wax_thickness,
        #type = 'heatmap',
        #colorscale = 'jet'
        name='wax_thickness2',
        legendgroup=legendgroup
    ),
    row=myrow,col=1,
)

fig.add_trace(
    go.Scatter(
        x = slice_centers,
        y = df['px_wax_transverse_edges'].apply(lambda x: np.diff(x)[0]),
        #type = 'heatmap',
        #colorscale = 'jet'
        name='wax_width_px',
        legendgroup=legendgroup
    ),
    row=myrow,col=1,
)

fig.add_trace(
    go.Scatter(
        x = slice_centers,
        y = wax_top,
        #type = 'heatmap',
        #colorscale = 'jet'
        name='wax_top',
        legendgroup=legendgroup
    ),
    row=myrow,col=1,
)




legendgroup = 'areas';
myrow+=1;
fig.add_trace(
    go.Scatter(
        x = slice_centers,
        y = df['area_filled']-df['area'],
        #type = 'heatmap',
        #colorscale = 'jet'
        name='seg_area_holes',
        legendgroup=legendgroup
    ),
    row=myrow,col=1,
)

fig.add_trace(
    go.Scatter(
        x = slice_centers,
        y = df['area'],
        #type = 'heatmap',
        #colorscale = 'jet'
        name='seg_area',
        legendgroup=legendgroup
    ),
    row=myrow,col=1,
)

fig.add_trace(
    go.Scatter(
        x = slice_centers,
        y = df['area_filled'],
        #type = 'heatmap',
        #colorscale = 'jet'
        name='seg_area_filled',
        legendgroup=legendgroup
    ),
    row=myrow,col=1,
)

fig.add_trace(
    go.Scatter(
        x = slice_centers,
        y = df['area_convex'],
        #type = 'heatmap',
        #colorscale = 'jet'
        name='seg_area_convex',
        legendgroup=legendgroup
    ),
    row=myrow,col=1,
)


legendgroup = 'ratios';
myrow+=1;
fig.add_trace(
    go.Scatter(
        x = slice_centers,
        y = df['area']/df['area_filled'],
        #type = 'heatmap',
        #colorscale = 'jet'
        name='seg_area_to_areafilled_ratio',
        legendgroup=legendgroup
    ),
    row=myrow,col=1,
)
fig.add_trace(
    go.Scatter(
        x = slice_centers,
        y = df['area']/df['area_convex'],
        #type = 'heatmap',
        #colorscale = 'jet'
        name='seg_area_to_areaconvex_ratio',
        legendgroup=legendgroup
    ),
    row=myrow,col=1,
)
fig.add_trace(
    go.Scatter(
        x = slice_centers,
        y = df['area_filled']/df['area_convex'],
        #type = 'heatmap',
        #colorscale = 'jet'
        name='seg_areafilled_to_areaconvex_ratio',
        legendgroup=legendgroup
    ),
    row=myrow,col=1,
)

legendgroup = 'counts';
myrow+=1;
fig.add_trace(
    go.Scatter(
        x = slice_centers,
        y = df['num_paths'],
        name='crossovers_px_paths',
        legendgroup=legendgroup,
    ),
    row=myrow,col=1,
)
fig.update_yaxes(row=myrow,type='log')

# setup legends per row
for i, yaxis in enumerate(fig.select_yaxes(col=1), 1):
    legend_name = f"legend{i}"
    fig.update_layout({legend_name: dict(y=yaxis.domain[1], yanchor="top")}, showlegend=True)
    fig.update_traces(row=i, legend=legend_name)

fig.update_traces(xaxis='x{:}'.format(nrows))
fig.update_yaxes(title='pixels',row=1);
fig.update_yaxes(title='pixels^2',row=2);
fig.update_yaxes(title='ratios',row=3);
fig.update_yaxes(title='counts',row=4);
fig.update_xaxes(title='distance along strip (px)',row=nrows,col=1);
fig.update_layout(hovermode='x unified',hoversubplots="axis",spikedistance=-1);
#fig = px.imshow(sum_wax_strip_along_length[:,0:]);

fig.show(renderer='browser')
#fig